# SAOS data quality

What is wrong, missing or misleading in the SAOS dump, measured on the interim tables (`records`, `cases`, `pairs`, `labels`) and the frozen datasets. Each check reports a count, its denominator, an example, and a treatment status. The date-anomaly count retains the three corrected source typos for audit. The table is saved to `results/saos-data-quality.csv`.

Reading `records.parquet` takes a minute; only metadata columns are loaded.

In [ ]:
import pandas as pd

from saos_appeals import config, eda, quality

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 100)

tables = quality.load()
{k: len(getattr(tables, k)) for k in ("records", "cases", "pairs", "labels", "datasets")}

## All checks

In [ ]:
report = quality.run_all(tables)
report.to_csv(config.RESULTS_DIR / "saos-data-quality.csv", index=False, lineterminator="\n")
report.drop(columns="example")

In [ ]:
report[["issue", "example"]]

## 1. Judgments without text

A record can exist in SAOS with an empty `textContent`. These are useless for us and they are not spread evenly over time.

In [ ]:
by_year = quality.empty_text_by_year(tables)
eda.plot_counts(
    by_year["share"],
    "share of SENTENCE records with empty text, by judgment year",
    "judgment year",
)
by_year.tail(12)

## 2. SAOS' own cross-references are not reliable

SAOS links judgments by matching signatures (`judgmentIds` in `referencedCourtCases`). A signature such as `I C 312/12` exists in every court, so most links point at a case in an unrelated court. This is why `saos-pair` ignores them and matches through the court hierarchy instead.

In [ ]:
report[report["issue"].str.startswith("SAOS judgmentIds")][["issue", "count", "share"]]

## 3. First-instance judgments are published selectively

For most appeals the judgment under review is simply not in SAOS. Coverage differs a lot between appellate courts, which is a selection effect any model trained on this data inherits.

In [ ]:
cov = quality.coverage_by_court(tables, min_appeals=200)
eda.plot_granted(
    cov.rename(columns={"coverage": "granted", "appeals": "n"}),
    "share of appeals whose first-instance judgment is in SAOS (≥ 200 appeals)",
    xlabel="share of appeals with the first-instance judgment in SAOS",
)
cov

## 4. Pairing losses step by step

Where the appeals go between 'has a text' and 'has a confirmed first-instance pair'.

In [ ]:
report[
    report["issue"].isin(
        [
            "appeal sentence without text",
            "appeal text without a recognisable header",
            "header without a first-instance signature",
            "first-instance judgment not published in SAOS",
            "matched by signature but header date or court disagrees",
        ]
    )
][["issue", "count", "out_of", "share", "note"]]

## 5. Dates, signatures, judges

Small in number but worth knowing: typo years, reasons dated long after the ruling, non-standard signatures, anonymised or merged judge names, signatures filed in the wrong kind of division.

In [ ]:
mask = report["issue"].str.contains("date|signature|SENTENCE record|judge|division", case=False)
report[mask][["issue", "count", "out_of", "share", "note", "example"]]

## 6. Text problems

Transcripts instead of judgments, typos that defeat the label regex, and anonymisation density in the reasons we feed to the model.

In [ ]:
mask = report["issue"].str.contains("transcript|typo|empty|anonymis|shorter")
report[mask][["issue", "count", "out_of", "share", "note"]]

In [ ]:
d = tables.datasets
d["anon_per_1k"] = 1000 * d["anon_tokens"] / d["first_reasons_len"].clip(lower=1)
d.groupby("domain")["anon_per_1k"].describe(percentiles=[0.5, 0.9]).round(1)

## What this means for the project

- **Coverage, not noise, is the main limitation.** Nine in ten appeals lose their first-instance judgment because it was never published; the remaining rows are a court-skewed sample.
- **Do not trust SAOS links.** Pair through the court hierarchy, court name and date from the header.
- **Expect the source to be dirty in small ways everywhere:** typo years, duplicated records, empty texts, transcripts. Each one needs a filter, and each filter is a line in `saos-dataset`.
- **Anonymisation is heavy and uneven.** `(...)` replaces names, companies and sometimes amounts; a text model sees this as a frequent token and should not learn from it.